In [0]:
-- VIEWS
/*
 Views in SQL is a VIRTUAL TABLE created using SQL Query. It allows us to create a table based on the result of a SQL query and can be used whenever we need the data.
Views are useful for the following reasons:
    1. Simplify complex queries
    2. Hide complexity of data
    3. Security
    4. Data Consistency

VIEWS are used to CREATE DataMart.

NOTE: 
    * Views itseld doesn't store the data rather an SQL Query and whnever we need the data it executes the Stored Query.
    * We can also Perform CRUD ops theougj Views
    * WITH CHECK OPTION ensures that INSERT and UPDATE operations performed through a view satisfy the view's filtering condition.
    * We can also UPDATE the view defintion with ALTER VIEW View_Name AS / CREATE OR REPLACE VIEW View_Name AS
    * We can also DROP the view with DROP VIEW.
    * A view can be created wit another view
*/

-- MATERIALIZED VIEW vs NORMAL VIEW
/*
    A normal view generally does not store the query result physically. It stores the query definition.
    A materialized view stores the result of the query physically and must be refreshed to reflect changes in its source data.
*/

-- find the running total of sales 
WITH CTE_Monthly_Summary AS
(
    SELECT
        date_format(orderDate, 'MMMM') as orderMonth,
        month(orderDate) as monthnum,
        count(productid) AS totalOrders,
        SUM(sales) as totalSales
    FROM workspace.default.sales
    GROUP BY orderMonth, monthnum
    ORDER BY monthnum
)
SELECT
    orderMonth,
    totalSales,
    SUM(totalSales) OVER (ORDER BY monthnum) as runningTotal,
    totalorders
FROM CTE_Monthly_Summary;

-- CREATE VIEW
CREATE VIEW workspace.default.V_Monthly_Summary AS
(
     SELECT
        date_format(orderDate, 'MMMM') as orderMonth,
        month(orderDate) as monthnum,
        count(productid) AS totalOrders,
        SUM(sales) as totalSales
    FROM workspace.default.sales
    GROUP BY orderMonth, monthnum
    ORDER BY monthnum
)

-- We can run View independent;y unlike CTE
SELECT 
    orderMonth,
    totalSales,
    totalOrders
FROM workspace.default.V_Monthly_Summary
WHERE totalOrders > 1;

-- update the view
CREATE OR REPLACE VIEW workspace.default.V_Monthly_Summary AS
(
     SELECT
        date_format(orderDate, 'MMMM') as orderMonth,
        month(orderDate) as monthnum,
        count(productid) AS totalOrders,
        SUM(sales) as totalSales
    FROM workspace.default.sales
    WHERE orderStatus = 'delivered'
    GROUP BY orderMonth, monthnum
    ORDER BY monthnum
)

SELECT 
    orderMonth,
    totalSales,
    totalOrders
FROM workspace.default.V_Monthly_Summary;


-- update productid from 101 to 1, 102 to 2 ans so on in sales table
-- UPDATE workspace.default.sales
-- SET productid = CASE productid
--         WHEN 101 THEN 1
--         WHEN 102 THEN 2
--         WHEN 103 THEN 3
--         WHEN 104 THEN 4
--         WHEN 105 THEN 5
--         WHEN 106 THEN 6
--     END

-- combine order, sales, details table
CREATE VIEW workspace.default.V_sales AS
(
    SELECT
        productid,
        productname
    FROM workspace.default.sales
    GROUP BY productid, productname
);

CREATE VIEW V_orderStatus AS
(
    SELECT 
        productid,
        orderStatus
    FROM workspace.default.sales
    GROUP BY productid, orderStatus
    HAVING orderStatus = 'pending'
    OR orderStatus = 'shipped'
);

CREATE OR REPLACE VIEW V_Combined AS
(
    SELECT  
        o.order_id,
        o.order_date,
        d.name,
        vs.productname,
        c.country,
        vos.orderStatus
    FROM workspace.default.orders o
    LEFT JOIN workspace.default.details d
    ON o.cust_id = d.emp_id
    INNER JOIN workspace.default.V_sales vs
    ON d.emp_id = vs.productid
    LEFT JOIN workspace.default.customers c
    ON vs.productid = c.cust_id
    INNER JOIN v_orderStatus vos
    ON c.cust_id = vos.productid
    ORDER BY o.order_id
);

SELECT *
FROM V_Combined;

-- DROP All the created views
DROP VIEW workspace.default.V_Monthly_Summary;
DROP VIEW workspace.default.V_sales;
DROP VIEW V_orderStatus;
DROP VIEW V_Combined;






